<a href="https://colab.research.google.com/github/francji1/01RAD/blob/main/HW/01RAD_Ex02_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01RAD - Homework 02: Regression to the mean (Galton's heights)

In 1886 Francis Galton compared the heights of 1,078 fathers and their adult sons and noticed that sons of tall fathers tend to be shorter than their fathers, and sons of short fathers taller. He called the phenomenon *regression towards mediocrity*; the name stuck to the whole method (Lecture 1, history). The data are the `fsdata.csv` file from Exercise 01 (heights in inches).

Model of Lecture 1: $Y_i = \beta_0 + \beta_1 x_i + e_i$ with $x_i$ the father's height and $Y_i$ the son's height, independent errors $e_i \sim (0, \sigma^2)$. Tools of Lecture 2 (slides pp. 14–32 of the lecture PDF): standard errors $\widehat{\sigma}(\widehat{\beta}_j) = s_n \sqrt{\delta_j}$, **confidence intervals** (Czech: intervaly spolehlivosti) $\widehat{\beta}_j \pm t_{1-\alpha/2}(n-2)\,\widehat{\sigma}(\widehat{\beta}_j)$ and the **tests** (testy hypotéz) with statistics $T_j = (\widehat{\beta}_j - b)/\widehat{\sigma}(\widehat{\beta}_j)$ compared with $t_{1-\alpha/2}(n-2)$.

## Submission
Work through the tasks in this notebook (code plus short written answers in markdown cells; Python or R, as you prefer). Submit through GitHub **before the next exercise**: fork `francji1/01RAD`, save the notebook into the `HW/` folder as `01RAD_Ex02_HW_<Surname>.ipynb`, with your surname in place of `<Surname>` and without the brackets, for example `01RAD_Ex02_HW_Novak.ipynb` (team: surnames joined by an underscore), and open a pull request to `main`. The pull request should add your one file only; do not modify or delete other files. A selected student solution, and a reference solution if needed, will be published in `HW/` a week later.

Useful calls: `stats.t.ppf(1 - alpha/2, df)` for the $t$ quantile, `stats.t.sf(abs(T), df)` and `stats.t.cdf(T, df)` for tail probabilities, `model.conf_int(alpha)`, `model.t_test('father = 1')`, `model.predict(pd.DataFrame({'father': [...]}))`, `np.corrcoef`, `df.sample(n, random_state=...)`.

## Data

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', palette='deep')

url_fsdata = 'https://raw.githubusercontent.com/francji1/01RAD/main/data/fsdata.csv'
fsdata = pd.read_csv(url_fsdata, index_col=0)
print(f'Rows: {fsdata.shape[0]}, columns: {fsdata.shape[1]}')
fsdata.describe().T

Rows: 1078, columns: 2


,count,mean,std,min,25%,50%,75%,max
father,1078.0,67.687097,2.744868,59.00800,65.787735,67.76660,69.60298,75.43393
son,1078.0,68.684070,2.814702,58.50708,66.931232,68.61582,70.46597,78.36479


## Task 01 - Fit and recap
Fit the regression of `son` on `father`. Compute $\widehat{\beta}_0$, $\widehat{\beta}_1$, $s_n$ and both standard errors by hand (formulas of Lectures 1 and 2) and check them against `statsmodels` (R: `summary(lm(son ~ father))`). Interpret $\widehat{\beta}_1$ in one sentence.

In [3]:
n = len(fsdata)
x = fsdata['father']
y = fsdata['son']

# Manual calculations
x_bar, y_bar = x.mean(), y.mean()
S_xx = np.sum((x - x_bar)**2)
S_yy = np.sum((y - y_bar)**2)
S_xy = np.sum((x - x_bar) * (y - y_bar))

beta_1_hat = S_xy / S_xx
beta_0_hat = y_bar - beta_1_hat * x_bar

# Residuals and variance
y_hat = beta_0_hat + beta_1_hat * x
RSS = np.sum((y - y_hat)**2)
s_n_sq = RSS / (n - 2)
s_n = np.sqrt(s_n_sq)

# Standard errors
se_beta_1 = s_n / np.sqrt(S_xx)
se_beta_0 = s_n * np.sqrt(1/n + x_bar**2 / S_xx)

print("--- Manual Calculations ---")
print(f"Beta_0_hat: {beta_0_hat:.4f}, SE: {se_beta_0:.4f}")
print(f"Beta_1_hat: {beta_1_hat:.4f}, SE: {se_beta_1:.4f}")
print(f"Residual standard error (s_n): {s_n:.4f}\n")

# Statsmodels for comparison
model = smf.ols('son ~ father', data=fsdata).fit()
print("--- Statsmodels Summary ---")
print(model.summary().tables[1])

--- Manual Calculations ---
Beta_0_hat: 33.8866, SE: 1.8324
Beta_1_hat: 0.5141, SE: 0.0270
Residual standard error (s_n): 2.4366

--- Statsmodels Summary ---
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     33.8866      1.832     18.493      0.000      30.291      37.482
father         0.5141      0.027     19.006      0.000       0.461       0.567


For every 1 inch increase in the father's height, the son's height is expected to increase by approximately 0.514 inches on average.

## Task 02 - Confidence intervals
Compute the 95 % and the 99 % confidence intervals for $\beta_0$ and $\beta_1$ by hand, using the $t$ quantile, and compare with `conf_int` (R: `confint`). Why is the 99 % interval wider? Compare the widths of the intervals for $\beta_0$ and $\beta_1$ in absolute terms and relative to the estimates: which parameter is estimated more precisely? Relate your answer to the remark in Lecture 2.

In [9]:
t_95 = stats.t.ppf(0.975, df=n-2)
t_99 = stats.t.ppf(0.995, df=n-2)

# 95% CI
ci_95_beta_0 = (beta_0_hat - t_95 * se_beta_0, beta_0_hat + t_95 * se_beta_0)
ci_95_beta_1 = (beta_1_hat - t_95 * se_beta_1, beta_1_hat + t_95 * se_beta_1)

# 99% CI
ci_99_beta_0 = (beta_0_hat - t_99 * se_beta_0, beta_0_hat + t_99 * se_beta_0)
ci_99_beta_1 = (beta_1_hat - t_99 * se_beta_1, beta_1_hat + t_99 * se_beta_1)

print("--- Manual CIs ---")
print(f"95% CI Beta_0: {ci_95_beta_0}")
print(f"95% CI Beta_1: {ci_95_beta_1}")
print(f"99% CI Beta_0: {ci_99_beta_0}")
print(f"99% CI Beta_1: {ci_99_beta_1}")

print("\n--- Statsmodels CIs ---")
print("\n--- 0.95% ---")
print(model.conf_int(alpha=0.05))
print("\n--- 0.99% ---")
print(model.conf_int(alpha=0.01))


--- Manual CIs ---
95% CI Beta_0: (np.float64(30.291212566757814), np.float64(37.481996141398035))
95% CI Beta_1: (np.float64(0.46101878906416827), np.float64(0.5671672881824467))
99% CI Beta_0: (np.float64(29.158387137657833), np.float64(38.614821570498016))
99% CI Beta_1: (np.float64(0.44429631121264995), np.float64(0.583889766033965))

--- Statsmodels CIs ---

--- 0.95% ---
                   0          1
Intercept  30.291213  37.481996
father      0.461019   0.567167

--- 0.99% ---
                   0          1
Intercept  29.158387  38.614822
father      0.444296   0.583890


In [14]:
# absolute comparison
Beta_0_width = ci_95_beta_0[1] - ci_95_beta_0[0]
Beta_1_width = ci_95_beta_1[1] - ci_95_beta_1[0]

#relative comparison
Beta_0_width_rel = Beta_0_width / beta_0_hat
Beta_1_width_rel = Beta_1_width / beta_1_hat

print("--- Absolute comparison ---")
print(f"Beta_0 width: {Beta_0_width:.4f}")
print(f"Beta_1 width: {Beta_1_width:.4f}")

print("\n--- Relative comparison ---")
print(f"Beta_0 width: {Beta_0_width_rel:.4f}")
print(f"Beta_1 width: {Beta_1_width_rel:.4f}")

--- Absolute comparison ---
Beta_0 width: 7.1908
Beta_1 width: 0.1061

--- Relative comparison ---
Beta_0 width: 0.2122
Beta_1 width: 0.2065


**Why is the 99% interval wider?**

To have higher confidence that the interval contains the true parameter, the interval must cover a larger range of possible values.


**Which parameter is estimated more precisely?**

 The absolute width for $\beta_0$ is much larger. $\beta_1$ is estimated more precisely relative to its size (the interval width relative to estimate size is smaller). Remark from lecture: it is usually the case that the confidence interval is wider for $\beta_0$ than for $\beta_1$, especially when the data is far from the origin.



## Task 03 - Is the father's height informative?
Test $H_0: \beta_1 = 0$ against $H_1: \beta_1 \neq 0$ at $\alpha = 0.05$: report $T_1$, the critical value $t_{0.975}(n-2)$, the $p$-value and the decision. What does the rejection say about the relationship, and what does it *not* say (think of the lecture's warning about linearity)?

In [17]:
T_1 = beta_1_hat / se_beta_1
crit_val = stats.t.ppf(0.975, df=n-2)
p_value = 2 * (1 - stats.t.cdf(np.abs(T_1), df=n-2))

print(f"Test statistic T_1: {T_1:.4f}")
print(f"Critical value t_0.975(n-2): {crit_val:.4f}")
print(f"P-value: {p_value:.4e}")

# Or this could also be done via the statsmodels command from Task 01
model = smf.ols('son ~ father', data=fsdata).fit()
print("--- Statsmodels Summary ---")
print(model.summary().tables[1])

Test statistic T_1: 19.0062
Critical value t_0.975(n-2): 1.9622
P-value: 0.0000e+00
--- Statsmodels Summary ---
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     33.8866      1.832     18.493      0.000      30.291      37.482
father         0.5141      0.027     19.006      0.000       0.461       0.567


**Decision:**

Since p-value 0.00 < 0.05, we reject the null hypothesis $H_0$: $\beta_1$ = 0.

**Interpretation:**

Father's height is significantly informative for predicting son's height. However, a significant linear relationship does not guarantee that the relationship is strictly linear across all values, nor does it imply causality.

## Task 04 - Regression to the mean
If sons were on average exactly as tall as their fathers plus a constant, the slope would be $\beta_1 = 1$. Test $H_0: \beta_1 = 1$ against $H_1: \beta_1 \neq 1$ (the test with a given candidate value $b_1$ from Lecture 2) and also against the **one-sided alternative** (jednostranná alternativa) $H_1: \beta_1 < 1$: the lecture treats two-sided alternatives, for the one-sided one use the same statistic $T = (\widehat{\beta}_1 - 1)/\widehat{\sigma}(\widehat{\beta}_1)$, reject when $T < -t_{1-\alpha}(n-2)$, and the $p$-value is $P(t(n-2) < T)$. Check the duality of the two-sided test with the 95 % confidence interval from Task 02. Explain in your own words why Galton spoke of *regression* (regrese k průměru) and what the estimated slope says about sons of very tall and very short fathers.

In [18]:
# Two-sided test H0: Beta_1 = 1 vs H1: Beta_1 != 1
T_stat = (beta_1_hat - 1) / se_beta_1
p_val_two_sided = 2 * (1 - stats.t.cdf(np.abs(T_stat), df=n-2))

# One-sided test H0: Beta_1 = 1 vs H1: Beta_1 < 1
p_val_one_sided = stats.t.cdf(T_stat, df=n-2) # Since T_stat is negative

print(f"Test statistic T for Beta_1 = 1: {T_stat:.4f}")
print(f"Two-sided p-value: {p_val_two_sided:.4e}")
print(f"One-sided p-value: {p_val_one_sided:.4e}")


Test statistic T for Beta_1 = 1: -17.9641
Two-sided p-value: 0.0000e+00
One-sided p-value: 1.3048e-63


**Decision:**

Reject both the two-sided and one-sided hypothesis $H_0$ at level of significance $\alpha$ = 0.05 as both p-values are strictly lesser than 0.05.


**Duality with CI from Task 02**

The 95% CI for $\beta_1$ from Task 02 does not contain 1, which perfectly aligns with rejecting H0: $\beta_1$ = 1 at $\alpha$ = 0.05.


**Regression towards mediocrity:**

The estimated slope (approx 0.51) is less than 1, with a positive intercept (approx. 34). This means very tall fathers (approx. twice as tall as the intercept) tend to have sons who are shorter than them and very short fathers (shorter than approx. twice the intercept) tend to have sons who are taller than them, 'regressing' towards the average height of the population.

## Task 05 - The intercept
Test $H_0: \beta_0 = 0$. Then write $\widehat{\beta}_0 = \overline{y} - \widehat{\beta}_1 \overline{x}$, plug in the sample means and explain why the intercept is far from zero even though sons are on average only about one inch taller than their fathers. Would a model through the origin make sense here?

In [19]:
# Task 05 - The intercept
T_0 = beta_0_hat / se_beta_0
p_val_0 = 2 * (1 - stats.t.cdf(np.abs(T_0), df=n-2))

print(f"Test statistic T_0: {T_0:.4f}, p-value: {p_val_0:.4e}")
print(f"Mean Father (x_bar): {x_bar:.2f}, Mean Son (y_bar): {y_bar:.2f}")
print(f"Beta_0_hat = {y_bar:.2f} - ({beta_1_hat:.4f} * {x_bar:.2f}) = {beta_0_hat:.2f}")

Test statistic T_0: 18.4935, p-value: 0.0000e+00
Mean Father (x_bar): 67.69, Mean Son (y_bar): 68.68
Beta_0_hat = 68.68 - (0.5141 * 67.69) = 33.89


**Why is the intercept far from zero?**

 The intercept represents the theoretical height of a son whose father is 0 inches tall. Since the observed heights are clustered around 65-70 inches, the regression line projects backward to a high y-intercept to fit the gentle slope through the data cloud.


**Would a model through the origin make sense?**

 No. A model through the origin forces $\beta_0$ = 0, implying a 0-inch father has a 0-inch son, but this heavily skews the slope for the realistic range of human heights. Unrealistic estimates close to the origin are conversely not a problem because humans that short are also unrealistic (or pathological cases).

## Task 06 - Point predictions
Predict the mean height of sons whose fathers are 60, 64, 68, 72 and 76 inches tall and tabulate the difference *predicted son minus father*. For which father's height do the two coincide? (Confidence and prediction intervals for these predictions come in Lecture 4; here point predictions are enough.)

In [ ]:
# your solution


## Task 07 - Reverse regression
Fit the reverse regression of `father` on `son` and report its slope. Verify numerically that the product of the two slopes equals the square of the sample correlation coefficient $r$ of the two heights (`np.corrcoef`), and explain why both slopes are smaller than 1. Is regression to the mean a causal effect of fathers on sons, or a statistical property of the two distributions?

In [ ]:
# your solution


## Task 08 - Sample size and precision
Draw random subsamples of 30, 100 and 300 father-son pairs (fix the random seed), fit the model in each and compare the widths of the 95 % intervals for $\beta_1$ with the full-sample interval. Explain the pattern through $\mathrm{Var}(\widehat{\beta}_1) = \sigma^2 / S_{xx}$. *Optional:* repeat the subsampling with $n = 100$ five hundred times, treat the full-sample $\widehat{\beta}_1$ as the true value and estimate how often the 95 % interval covers it.

In [ ]:
# your solution
